# 第5日 ノートブック02：PatchCore を学習させる

# Day 5 Notebook 02: Training PatchCore

---

## 学習目標 / Learning objectives

| 日本語 | English |
| --- | --- |
| 1. Anomalib の API を使って PatchCore を初期化できる | 1. Initialise PatchCore through the Anomalib API |
| 2. 異常検知用のデータ構成を説明できる | 2. Explain the dataset layout used for anomaly detection |
| 3. 正常画像だけで学習（fit）できる | 3. Fit the model on normal images only |
| 4. 異常スコアとヒートマップを取得できる | 4. Obtain anomaly scores and heatmaps |
| 5. 結果を保存し、しきい値を検討できる | 5. Save results and reason about the threshold |

## このノートブックの基本情報 / Notebook facts

| 項目 / Item | 内容 / Details |
| --- | --- |
| 学習時間（目安） / Estimated duration | 150 分 / 150 minutes |
| 難易度 / Difficulty | 初級 / Beginner |
| 前提条件 / Prerequisites | 第5日ノートブック01 を終え、`aivision-anomalib` カーネルを選択していること。 / You finished Day 5 Notebook 01 and selected the `aivision-anomalib` kernel. |
| 必要なライブラリ / Required libraries | anomalib 2.7.0, torch, torchvision, opencv-python-headless, numpy, matplotlib, pandas |
| 使用カーネル / Jupyter kernel | `aivision-anomalib`（**必ずこのカーネルを使ってください**） |

### 提出物 / Expected deliverables

| 日本語 | English |
| --- | --- |
| 学習結果（指標） | Training results (metrics) |
| ヒートマップ画像 | Heatmap images |
| 画像ごとの異常スコアの表 | A table of per-image anomaly scores |

> **English:** This notebook **requires the `aivision-anomalib` kernel**. Week 1 uses two separate environments because Anomalib requires `opencv-python-headless` while Ultralytics requires `opencv-python`; installing both into one environment breaks OpenCV.

## 本日の学習タイムライン / Learning timeline for today

| 時刻 / Time | 内容（日本語） | Activity (English) |
| --- | --- | --- |
| 08:00–09:00 | 講師：デモ（PatchCore の学習） | Instructor: live PatchCore demo |
| 09:00–10:30 | Part 1〜Part 3（環境・データ・初期化） | Part 1-Part 3 (environment, data, model) |
| 10:30–10:40 | 休憩 | Short break |
| 10:40–12:00 | Part 4〜Part 5（学習・推論・可視化） | Part 4-Part 5 (fit, predict, visualise) |
| 12:00–12:50 | 昼休み | Lunch break |
| 12:50–13:50 | 講師：結果の確認と午後の説明 | Instructor: results review and briefing |
| 13:50–16:50 | 演習・提出物 | Exercises and submission |

> **日本語：** 12:00–12:50 は昼休みです。08:00–09:00 と 12:50–13:50 は講師の時間、それ以外は自分のペースで進めてください。
>
> **English:** 12:00-12:50 is the lunch break. 08:00-09:00 and 12:50-13:50 are instructor sessions; the remaining time is self-paced work.

---

## Part 0　準備 / Preparation

In [ ]:
from pathlib import Path

def find_course_root(start=None):
    """研修フォルダ（AI_Vision_Training）を上方向に探す。
    Walk upwards until the course root folder is found."""
    current = Path(start).resolve() if start else Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "assets").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError(
        "研修フォルダが見つかりません。JupyterLab を AI_Vision_Training で起動してください。\n"
        "Course folder not found. Start JupyterLab inside the AI_Vision_Training folder."
    )

COURSE_ROOT = find_course_root()
ASSETS = COURSE_ROOT / "assets"
print("研修フォルダ / course root :", COURSE_ROOT)
print("素材フォルダ / assets      :", ASSETS)

#### 日本語：カーネルを確認する
**日本語：** このノートブックは **`aivision-anomalib` カーネル** で実行します。
違うカーネルだと `ModuleNotFoundError: No module named 'anomalib'` になります。

その場合は メニューの `Kernel` → `Change Kernel` → `Python 3 (AI Vision Anomalib)` を選び、
**ノートブックを最初から実行し直してください**。

**English:** Run this notebook with the **`aivision-anomalib`** kernel. With another kernel you get
`ModuleNotFoundError: No module named 'anomalib'`.

Choose `Kernel` -> `Change Kernel` -> `Python 3 (AI Vision Anomalib)` and **re-run from the top**.
#### English: Confirm the kernel

In [ ]:
import importlib
import sys

print("Python version:", sys.version.split()[0])

REQUIRED = [
    ("anomalib", "anomalib"),
    ("torch", "torch"),
    ("torchvision", "torchvision"),
    ("cv2", "opencv-python-headless"),
    ("lightning", "lightning"),
    ("pandas", "pandas"),
]

missing = []
for import_name, pip_name in REQUIRED:
    try:
        module = importlib.import_module(import_name)
        version = getattr(module, "__version__", "unknown")
        print(f"  [OK]      {pip_name:<20} {version}")
    except ImportError:
        print(f"  [MISSING] {pip_name:<20} -> pip install {pip_name}")
        missing.append(pip_name)

if missing:
    print("\n未導入のパッケージがあります / Some packages are missing:", ", ".join(missing))
    print("docs/troubleshooting/installation.md を確認してください / see that file for help.")
else:
    print("\nこのノートブックに必要なライブラリはすべて揃っています。")
    print("All libraries required by this notebook are available.")

In [ ]:
import platform
from pathlib import Path

from matplotlib import font_manager, rcParams

def find_japanese_font():
    """日本語を表示できるフォントを探す。見つからなければ None を返す。
    Locate a font that can render Japanese. Returns None when nothing is found."""
    system = platform.system()
    if system == "Windows":
        folders = [Path("C:/Windows/Fonts")]
    elif system == "Darwin":
        folders = [Path("/System/Library/Fonts"), Path("/Library/Fonts"), Path.home() / "Library/Fonts"]
    else:
        folders = [Path("/usr/share/fonts"), Path("/usr/local/share/fonts"), Path.home() / ".fonts"]

    preferred = [
        "NotoSansJP-Regular", "NotoSansCJK-Regular", "YuGothR", "meiryo", "msgothic",
        "BIZ-UDGothicR", "NotoSansJP", "HiraginoSans",
    ]
    found = []
    for folder in folders:
        if not folder.is_dir():
            continue
        for path in folder.rglob("*"):
            if path.suffix.lower() not in {".ttf", ".ttc", ".otf"}:
                continue
            name = path.stem
            if any(key.lower() in name.lower() for key in preferred):
                found.append(path)

    def rank(path):
        name = path.stem.lower()
        return (0 if "regular" in name or name.endswith("r") else 1, len(name))

    return sorted(found, key=rank)[0] if found else None


japanese_font = find_japanese_font()
if japanese_font is None:
    print("日本語フォントが見つかりません / no Japanese font found.")
    print("図のタイトルが □ になる場合は、英字のタイトルを使ってください。")
    print("If figure titles show boxes, use English titles instead.")
else:
    font_manager.fontManager.addfont(str(japanese_font))
    family_name = font_manager.FontProperties(fname=str(japanese_font)).get_name()
    rcParams["font.family"] = [family_name, "DejaVu Sans"]
    rcParams["axes.unicode_minus"] = False
    print("図の日本語フォント / figure font:", family_name)
    print("  ->", japanese_font)

In [ ]:
import warnings
warnings.filterwarnings("ignore")     # 依存ライブラリの警告を抑える / silence dependency warnings

import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import cv2
import matplotlib.pyplot as plt

import anomalib
from anomalib.data import Folder
from anomalib.models import Patchcore
from anomalib.engine import Engine

print("anomalib   :", anomalib.__version__)
print("torch      :", torch.__version__)
print("CUDA       :", torch.cuda.is_available())
print("device     :", "cuda" if torch.cuda.is_available() else "cpu")

#### 日本語：使う API（バージョン固定）
**日本語：** Anomalib は **2.x 系** で API が大きく変わりました。
1.x 系のサンプルコードをコピーすると動きません。

この研修では **Anomalib 2.7.0** で動作確認しています。使うのは 3 つのクラスだけです。

| クラス | 役割 |
|---|---|
| `anomalib.data.Folder` | フォルダ構成からデータを読む |
| `anomalib.models.Patchcore` | PatchCore 本体 |
| `anomalib.engine.Engine` | 学習・評価・推論を実行する |

**English:** Anomalib's API changed substantially in **2.x**; copying 1.x sample code will not work.

This course is verified with **Anomalib 2.7.0**. Only three classes are used:

| Class | Role |
|---|---|
| `anomalib.data.Folder` | reads data from a folder layout |
| `anomalib.models.Patchcore` | the PatchCore model |
| `anomalib.engine.Engine` | runs fit, test and predict |
#### English: The API used here (pinned version)

---

## Part 1　データの構成 / The data layout

#### 日本語：正常だけを学習に使う
**日本語：** 使うフォルダ構成は次のとおりです。

```text
assets/anomaly_dataset/pvc_sheet/
    train/good/                      <- 学習に使う（正常のみ）
    test/good/                       <- 評価に使う（正常）
    test/black_spot/                 <- 評価に使う（異常）
    test/scratch/
    test/streak/
    test/irregular_mark/
    ground_truth/<種類>/xxx_mask.png <- 正解マスク（評価用）
```

**日本語：** **`train/` には異常画像が 1 枚も入っていません。** これが PatchCore の前提です。

**English:** The layout is:

```text
assets/anomaly_dataset/pvc_sheet/
    train/good/                      <- training (normal only)
    test/good/                       <- evaluation (normal)
    test/black_spot/                 <- evaluation (abnormal)
    test/scratch/, test/streak/, test/irregular_mark/
    ground_truth/<type>/xxx_mask.png <- ground-truth masks (evaluation)
```

**`train/` contains no abnormal images at all.** That is the premise of PatchCore.
#### English: Only normal images are used for training

In [ ]:
dataset_root = ASSETS / "anomaly_dataset" / "pvc_sheet"
layout_path = dataset_root / "dataset_layout.json"

if not layout_path.is_file():
    raise FileNotFoundError(
        "異常検知データセットがありません。PowerShell で次を実行してください:\n"
        "  python assets\\scripts\\prepare_datasets.py --only anomaly\n"
        "/ anomaly dataset missing; run prepare_datasets.py --only anomaly"
    )

layout = json.loads(layout_path.read_text(encoding="utf-8"))
print("dataset root :", dataset_root)
print("counts       :", layout["counts"])
print()
print("train に異常画像が無いことを確認する / confirm train has only normal images:")
print("  train/good/*.png :", len(list((dataset_root / "train" / "good").glob("*.png"))))
print("  train 直下の他のフォルダ / other folders under train:",
      [p.name for p in (dataset_root / "train").iterdir() if p.is_dir() and p.name != "good"])

#### 日本語：データモジュールを作る
**日本語：** `Folder` に、どのフォルダが何かを教えます。

| 引数 | 意味 |
|---|---|
| `normal_dir` | 学習に使う正常画像のフォルダ |
| `normal_test_dir` | 評価に使う正常画像のフォルダ |
| `abnormal_dir` | 評価に使う異常画像のフォルダ（リスト） |
| `mask_dir` | 正解マスクのフォルダ（リスト） |
| `val_split_mode` / `val_split_ratio` | 評価データを検証と試験に分ける |
| `num_workers` | Windows では 0 が安全 |

**日本語：** `val_split_mode="from_test"` にすると、評価データが **検証用と試験用に半分ずつ**
分かれます。**しきい値は検証用で決め、成績は試験用で測る** ためです。

**English:** `Folder` is told which folder is which.

| Argument | Meaning |
|---|---|
| `normal_dir` | normal images for training |
| `normal_test_dir` | normal images for evaluation |
| `abnormal_dir` | abnormal images for evaluation (list) |
| `mask_dir` | ground-truth mask folders (list) |
| `val_split_mode` / `val_split_ratio` | how the evaluation data is split into val/test |
| `num_workers` | 0 is safe on Windows |

With `val_split_mode="from_test"` the evaluation data is split in half: **the threshold is chosen on
val and the score is measured on test**.
#### English: Building the datamodule

In [ ]:
datamodule = Folder(
    name=layout["name"],
    root=str(dataset_root),
    normal_dir=layout["normal_dir"],
    normal_test_dir=layout["normal_test_dir"],
    abnormal_dir=layout["abnormal_dir"],
    mask_dir=layout["mask_dir"],
    train_batch_size=8,
    eval_batch_size=8,
    num_workers=0,                  # Windows のノートブックでは 0 / safe on Windows
    val_split_mode="from_test",     # 評価データを検証と試験に分ける
    val_split_ratio=0.5,
    seed=layout["seed"],
)
datamodule.setup()

print("学習サンプル / train :", len(datamodule.train_data))
print("検証サンプル / val   :", len(datamodule.val_data))
print("試験サンプル / test  :", len(datamodule.test_data))
print()
print("試験データの内訳 / test composition:")
print(datamodule.test_data.samples["label"].value_counts().to_string())

#### 日本語：学習データに異常が無いことを確認する
**日本語：** ここは **必ず確認** してください。`train` に異常画像が混ざると、
その異常は「正常」として覚えられ、**二度と検出できなくなります**。

**English:** **Always check this.** If abnormal images leak into the normal training set, they are
learned as "normal" and **can never be detected again**.
#### English: Verify training data contains no anomalies

In [ ]:
train_labels = datamodule.train_data.samples["label"].value_counts().to_dict()
print("train のラベル内訳 / train label counts:", train_labels)
assert len(train_labels) == 1 and list(train_labels)[0] == "normal", (
    "train に正常以外の画像が入っています / training data contains non-normal images"
)
print("OK: 学習データは正常のみです / training data is normal-only")

---

## Part 2　モデルを初期化する / Initialising the model

#### 日本語：バックボーンの選び方
**日本語：** `Patchcore` の主な引数です。

| 引数 | 意味 | この研修の値 |
|---|---|---|
| `backbone` | 特徴を計算するネットワーク | `resnet18`（CPU 向けに軽い） |
| `layers` | どの層の特徴を使うか | `["layer2", "layer3"]` |
| `pre_trained` | 学習済みの重みを使うか | `True` |
| `coreset_sampling_ratio` | メモリバンクに残す割合 | `0.1`（10 %） |
| `num_neighbors` | 距離計算に使う近傍数 | `9` |

**日本語：** 論文の既定は `wide_resnet50_2` ですが、**CPU では非常に遅い** ため、
この研修では `resnet18` を使います。**考え方は同じ** で、精度は少し下がります。

> **日本語：** `pre_trained=True` は、ImageNet で学習した重みを **初回だけダウンロード** します。
> ネットワークが必要です。取得できなかった場合は、そのことを記録して講師に相談してください。

**English:** The main `Patchcore` arguments:

| Argument | Meaning | Value here |
|---|---|---|
| `backbone` | network that computes features | `resnet18` (light, CPU friendly) |
| `layers` | which layers to use | `["layer2", "layer3"]` |
| `pre_trained` | use pretrained weights | `True` |
| `coreset_sampling_ratio` | fraction kept in the memory bank | `0.1` (10 %) |
| `num_neighbors` | neighbours used for the distance | `9` |

The paper's default is `wide_resnet50_2`, which is **far too slow on CPU**, so this course uses
`resnet18`. **The idea is identical**; accuracy is somewhat lower.

`pre_trained=True` **downloads ImageNet weights once**, so a network connection is needed. If it
fails, record it and ask the instructor.
#### English: Choosing the backbone

In [ ]:
model = Patchcore(
    backbone="resnet18",
    layers=["layer2", "layer3"],
    pre_trained=True,
    coreset_sampling_ratio=0.1,
    num_neighbors=9,
)
print("モデルを初期化しました / model initialised:", type(model).__name__)
print("内部モデル / inner model:", type(model.model).__name__)
print("coreset_sampling_ratio:", model.coreset_sampling_ratio)

---

## Part 3　学習（fit）する / Fitting the model

#### 日本語：PatchCore の「学習」は 1 回だけ
**日本語：** PatchCore の学習は、**ニューラルネットワークの重みを更新しません**。
やっていることは次の 2 つだけです。

1. 正常画像をバックボーンに通して **特徴を計算する**
2. その特徴を **メモリバンクに保存する**（コアセットで間引く）

**日本語：** だから `max_epochs=1` で十分です。**エポックを増やしても意味がありません。**
これは第 4 日の YOLO との大きな違いです。

**English:** Fitting PatchCore **does not update any neural-network weights**. It only:

1. runs the normal images through the backbone to **compute features**
2. **stores** those features in the memory bank (reduced to a coreset)

So `max_epochs=1` is enough. **More epochs mean nothing** — a key difference from YOLO on Day 4.
#### English: PatchCore's fit is a single pass

#### 日本語：経過表示を自分で出す理由
**日本語：** Lightning のリッチな経過表示は、Windows の Jupyter で
    ごく稀に `RecursionError` を起こすことがあります。この研修では
    **表示を切り替えて**、自分で開始・終了のメッセージを出します。
    学習そのものの動作は変わりません。

    **English:** Lightning's rich progress display can, rarely, raise `RecursionError` inside Jupyter
    on Windows. This course **switches it off** and prints its own start/finish messages. The training
    itself is unaffected.
#### English: Why the progress display is switched off

In [ ]:
import time

results_root = COURSE_ROOT / "training_results" / "day05"
engine = Engine(
    default_root_dir=str(results_root),
    accelerator="cpu",
    devices=1,
    max_epochs=1,
    logger=False,
    enable_progress_bar=False,   # Windows の Jupyter での表示不具合を避ける / avoid a Windows display issue
    enable_model_summary=False,
)

print("学習（fit）を開始します。1〜2 分かかります / fitting now, this takes 1-2 minutes...")
started = time.time()
engine.fit(model=model, datamodule=datamodule)
elapsed = time.time() - started
print()
print(f"学習（fit）が終わりました / fitting finished in {elapsed:.1f} seconds")

#### 日本語：保存されたもの
**日本語：** `engine.fit()` は、モデルの状態を `training_results/day05/` 以下に保存します。
**メモリバンクの中身は、この保存ファイルに入っています。**

**English:** `engine.fit()` writes the model state under `training_results/day05/`.
**The memory bank contents are part of that checkpoint.**
#### English: What was saved

In [ ]:
print("保存先 / output root:", results_root)
for path in sorted(results_root.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(results_root)}  ({path.stat().st_size} bytes)")

---

## Part 4　評価する / Evaluating

#### 日本語：4 つの指標
**日本語：** Anomalib は 4 つの指標を返します。

| 指標 | 意味 |
|---|---|
| `image_AUROC` | 画像単位の OK/NG 判別力（1.0 が最高） |
| `image_F1Score` | 画像単位の調和平均（precision と recall のバランス） |
| `pixel_AUROC` | 画素単位で異常領域を当てる力 |
| `pixel_F1Score` | 画素単位の調和平均 |

**日本語：** **AUROC** は「正常と異常を、しきい値に依存せずどれだけ分けられたか」を表します。
0.5 がでたらめ、1.0 が完璧です。

**English:** Anomalib reports four metrics:

| Metric | Meaning |
|---|---|
| `image_AUROC` | ability to separate OK/NG per image (1.0 is best) |
| `image_F1Score` | harmonic mean of image-level precision and recall |
| `pixel_AUROC` | ability to localise anomalous pixels |
| `pixel_F1Score` | harmonic mean of pixel-level precision and recall |

**AUROC** describes how well normal and abnormal separate **without fixing a threshold**:
0.5 is random, 1.0 is perfect.
#### English: The four metrics

In [ ]:
test_results = engine.test(model=model, datamodule=datamodule)

metrics = test_results[0] if isinstance(test_results, list) else test_results
metrics_frame = pd.DataFrame([metrics]).T
metrics_frame.columns = ["value"]
print(metrics_frame.round(4).to_string())

submission_dir = COURSE_ROOT / "submissions" / "day05"
submission_dir.mkdir(parents=True, exist_ok=True)
metrics_frame.to_csv(submission_dir / "patchcore_metrics.csv", encoding="utf-8-sig")
print()
print("保存 / saved:", submission_dir / "patchcore_metrics.csv")

### 実行結果の例 / Expected output

**日本語：** 次のような結果になります（数値は環境によって多少変わります）。

```text
value
image_AUROC    1.0000
image_F1Score  0.9744
pixel_AUROC    0.9841
pixel_F1Score  0.3857

保存 / saved: .../submissions/day05/patchcore_metrics.csv
```

**English:** You should see something like the following (exact numbers vary slightly between machines).

```text
image_AUROC    1.0000
image_F1Score  0.9744
pixel_AUROC    0.9841
pixel_F1Score  0.3857
```

#### 日本語：数字が良すぎるときに疑うこと
**日本語：** 合成画像は **実際の工場画像より簡単** です。欠陥がはっきりしており、
背景のばらつきも小さいため、`image_AUROC` が 1.0 に近づきます。

**これは「実力」ではありません。** 実際の PVC では、色ムラ・照明ムラ・反射などが加わり、
同じ数字は出ません。

**日本語：** この研修の数字を、**製品の検査精度として引用してはいけません。**

**English:** Synthetic images are **easier than real factory images**: defects are crisp and the
background variation is small, so `image_AUROC` approaches 1.0.

**That is not real-world skill.** Real PVC adds colour mottling, uneven lighting and reflections, and
the numbers will not transfer.

**Never quote this course's numbers as the inspection accuracy of a product.**
#### English: When the numbers look too good

---

## Part 5　異常スコアとヒートマップ / Scores and heatmaps

#### 日本語：1 枚ずつ結果を見る
**日本語：** `engine.predict()` は、画像ごとの結果を返します。

| 属性 | 内容 |
|---|---|
| `pred_score` | 異常スコア（0〜1） |
| `pred_label` | しきい値で決めた OK/NG |
| `anomaly_map` | 画素ごとの異常の度合い |
| `gt_label` | 正解（異常かどうか） |
| `image_path` | 画像のパス |

**English:** `engine.predict()` returns per-image results:

| Attribute | Content |
|---|---|
| `pred_score` | anomaly score (0-1) |
| `pred_label` | OK/NG decided by the threshold |
| `anomaly_map` | per-pixel anomaly strength |
| `gt_label` | ground truth (abnormal or not) |
| `image_path` | the image path |
#### English: Look at results image by image

In [ ]:
predictions = engine.predict(model=model, datamodule=datamodule)
print("バッチ数 / batches:", len(predictions))
print("最初のバッチのスコア / first batch scores:", predictions[0].pred_score[:5].tolist())

In [ ]:
rows = []
for batch in predictions:
    scores = batch.pred_score.detach().cpu().numpy()
    labels = batch.pred_label.detach().cpu().numpy().astype(bool)
    gt_labels = batch.gt_label.detach().cpu().numpy().astype(bool)
    for path, score, label, gt in zip(batch.image_path, scores, labels, gt_labels):
        rows.append({
            "image": Path(path).relative_to(dataset_root).as_posix(),
            "score": round(float(score), 4),
            "predicted_ng": bool(label),
            "truly_abnormal": bool(gt),
        })

scores_frame = pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)
print(scores_frame.head(10).to_string(index=False))
print()
print("件数 / rows:", len(scores_frame))

#### 日本語：数値を見るときの注意
**日本語：** `pred_label` は Anomalib が **検証データで決めたしきい値** による判定です。
しきい値は `post_processor` に入っており、`model.post_processor` から確認できます。

**日本語：** このしきい値は **合成データ用** です。実運用では、
**正常品だけを流してスコアの分布を測り、しきい値を決め直す** 必要があります。

**English:** `pred_label` uses **the threshold Anomalib chose on the validation data**, stored in the
`post_processor` and readable via `model.post_processor`.

That threshold is **for this synthetic data only**. In production you would **run known-good products,
measure the score distribution and set the threshold again**.
#### English: Caution when reading the numbers

In [ ]:
try:
    print("しきい値 / threshold:", float(model.post_processor.adaptive_threshold))
except Exception:
    print("post_processor からしきい値を取得できませんでした。")
    print("Could not read the threshold from the post_processor; inspect model.post_processor instead.")

print()
normal_scores = scores_frame.loc[~scores_frame["truly_abnormal"], "score"]
abnormal_scores = scores_frame.loc[scores_frame["truly_abnormal"], "score"]
print("正常品のスコア / normal score  : max =", round(float(normal_scores.max()), 4))
print("異常品のスコア / abnormal score: min =", round(float(abnormal_scores.min()), 4))
print("=> この 2 つの間にしきい値を置くと、両方を分けられます")
print("=> a threshold between these two separates both groups")

In [ ]:
# ヒートマップを可視化する / visualise the heatmaps
def show_heatmaps(batch, count=4):
    images = batch.image.detach().cpu().numpy()          # (N, 3, H, W), 0-1
    maps = batch.anomaly_map.detach().cpu().numpy()      # (N, H, W)
    scores = batch.pred_score.detach().cpu().numpy()
    truths = batch.gt_label.detach().cpu().numpy()

    count = min(count, len(images))
    fig, axes = plt.subplots(count, 3, figsize=(11, 3.1 * count))
    if count == 1:
        axes = axes.reshape(1, -1)

    for index in range(count):
        rgb = np.transpose(images[index], (1, 2, 0))
        heat = maps[index]
        axes[index, 0].imshow(rgb); axes[index, 0].set_title("image", fontsize=8)
        axes[index, 1].imshow(heat, cmap="inferno"); axes[index, 1].set_title("anomaly map", fontsize=8)
        axes[index, 2].imshow(rgb); axes[index, 2].imshow(heat, cmap="inferno", alpha=0.5)
        axes[index, 2].set_title(
            f"overlay  score={scores[index]:.3f}  truth={'NG' if truths[index] else 'OK'}", fontsize=8)
        for ax in axes[index]:
            ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()


for batch in predictions[:1]:
    show_heatmaps(batch, count=4)

#### 日本語：ヒートマップを画像として保存する
**日本語：** 第 5 日ノートブック 03 で、YOLO の結果と並べて表示するために、
ヒートマップを画像ファイルとして保存します。

**English:** Save the heatmaps as image files so Day 5 Notebook 03 can show them next to the YOLO
results.
#### English: Save the heatmaps as images

In [ ]:
heatmap_dir = submission_dir / "heatmaps"
heatmap_dir.mkdir(parents=True, exist_ok=True)

def save_overlay(batch, index, out_path):
    """1 枚分のヒートマップを重ねた画像を保存する / save one overlay image."""
    image = np.transpose(batch.image[index].detach().cpu().numpy(), (1, 2, 0))
    heat = batch.anomaly_map[index].detach().cpu().numpy()
    figure = plt.figure(figsize=(4, 4))
    plt.imshow(image)
    plt.imshow(heat, cmap="inferno", alpha=0.5)
    plt.axis("off")
    plt.tight_layout()
    figure.savefig(out_path, dpi=110, bbox_inches="tight")
    plt.close(figure)


saved_heatmaps = []
for batch in predictions:
    for index in range(len(batch.pred_score)):
        truth = bool(batch.gt_label[index])
        score = float(batch.pred_score[index])
        # 異常 1 枚につき 2 枚、正常は 1 枚まで保存する / save up to 2 abnormal and 1 normal
        if truth and sum(1 for h in saved_heatmaps if h["truth"]) >= 2:
            continue
        if (not truth) and sum(1 for h in saved_heatmaps if not h["truth"]) >= 1:
            continue
        name = Path(batch.image_path[index]).stem
        out_path = heatmap_dir / f"{name}_heatmap.png"
        save_overlay(batch, index, out_path)
        saved_heatmaps.append({"file": out_path.name, "score": round(score, 4), "truth": truth})

print(pd.DataFrame(saved_heatmaps).to_string(index=False))

#### 日本語：ヒートマップの読み方
**日本語：** 明るい（赤・黄色）ほど **異常の度合いが高い** 領域です。

- **NG の画像** … 欠陥の位置に明るい点が出る
- **OK の画像** … ほぼ全体が暗い（ムラが出ることはある）

**日本語：** もし OK の画像で明るい場所が出たら、それは
**「正常のばらつきを異常と判定している」** 可能性があります。
学習データに、そのばらつきを追加する必要があります。

**English:** Brighter (red/yellow) means **more anomalous**.

- **abnormal images**: a bright spot at the defect
- **normal images**: mostly dark (some mottling is acceptable)

If a normal image shows a bright region, the model is probably **calling acceptable variation an
anomaly**. Add that variation to the training data.
#### English: How to read a heatmap

In [ ]:
# 提出用にスコア一覧を保存する / save the score table for submission
scores_path = submission_dir / "patchcore_scores.csv"
scores_frame.to_csv(scores_path, index=False, encoding="utf-8-sig")

predictions_path = submission_dir / "patchcore_predictions.json"
predictions_path.write_text(
    json.dumps(
        {
            "note": (
                "PatchCore predictions on the Day 5 test split. "
                "Synthetic images, not real factory data."
            ),
            "model": "Patchcore(resnet18, layers=[layer2,layer3], coreset=0.1, neighbors=9)",
            "anomalib_version": anomalib.__version__,
            "metrics": {k: float(v) for k, v in metrics.items()},
            "images": rows,
        },
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("保存 / saved:", scores_path)
print("保存 / saved:", predictions_path)

---

## 演習 / Exercises

#### 日本語：演習 1（基本）：指標を記録する
**日本語：** Part 4 の 4 つの指標を表に記入してください。
`image_AUROC` と `pixel_AUROC` の違いも、1 行で説明してください。

**English:** Record the four metrics from Part 4 and explain in one line how `image_AUROC` differs
from `pixel_AUROC`.
#### English: Exercise 1 (basic): record the metrics
Done when all four values are recorded and the difference is explained.

In [ ]:
# 演習 1 / Exercise 1
record = {
    "image_AUROC": None,
    "image_F1Score": None,
    "pixel_AUROC": None,
    "pixel_F1Score": None,
    "difference": "",
}
for key, value in record.items():
    print(f"{key:<16} {value if value not in (None, '') else '(未記入 / not written)'}")

#### 日本語：演習 2（中級）：バックボーンを変える
**日本語：** `backbone="wide_resnet50_2"` に変えて、もう一度 fit と test を実行してください。
**指標** と **所要時間** を比べ、CPU で使うべきかを判断してください。

**English:** Switch to `backbone="wide_resnet50_2"`, fit and test again, then compare the **metrics**
and the **runtime** and judge whether it is usable on CPU.
#### English: Exercise 2 (intermediate): change the backbone
Done when both runs are recorded and you state which backbone you would choose.

In [ ]:
# 演習 2 / Exercise 2
import time

comparison = []
# 例 / example:
# started = time.time()
# model_wide = Patchcore(backbone="wide_resnet50_2", layers=["layer2","layer3"], pre_trained=True)
# engine_wide = Engine(default_root_dir=str(results_root / "wide"), accelerator="cpu", devices=1,
#                      max_epochs=1, logger=False)
# engine_wide.fit(model=model_wide, datamodule=datamodule)
# ... test ...
# comparison.append({"backbone": "wide_resnet50_2", "seconds": time.time() - started})

print(pd.DataFrame(comparison or [{"backbone": "(未実施 / not run)", "seconds": 0.0}]).to_string(index=False))

#### 日本語：演習 3（発展）：正常品のばらつきを増やす
**日本語：** `assets/scripts/pvc_synth.py` を使って、**明るさのムラを大きくした正常画像** を
10 枚追加で生成し、`train/good/` に置いてください。そのうえで再学習し、
**正常品のスコアが下がったか** を確認してください。

**English:** Using `assets/scripts/pvc_synth.py`, generate ten extra normal images with **stronger
brightness variation**, put them in `train/good/`, retrain, and check whether the **scores of normal
images fall**.
#### English: Exercise 3 (challenge): add normal variation
Done when ten extra images exist, training has been re-run, and the normal-score maximum is compared before and after.

In [ ]:
# 演習 3 / Exercise 3
import sys
sys.path.insert(0, str(ASSETS / "scripts"))
import pvc_synth as ps

extra_dir = dataset_root / "train" / "good"
before_max = float(normal_scores.max())
print("学習前の正常スコア最大 / before:", round(before_max, 4))

# 例 / example:
# for i in range(10):
#     image = ps.render_normal(256, 256, seed=90_000 + i, shading_strength=18.0, grain_strength=8.0)
#     ps.save_rgb(image, extra_dir / f"extra_variation_{i:03d}.png")
# print("追加しました / added:", len(list(extra_dir.glob('extra_variation_*.png'))))
print("追加したら、Part 3 の fit から再実行してください / then re-run from the Part 3 fit cell")

---

#### 日本語：理解度チェック / Understanding questions
**日本語：**

1. なぜ `train` に異常画像を入れてはいけないのですか。
2. PatchCore の「学習」は、何をしていますか。重みは更新されますか。
3. `image_AUROC` と `pixel_AUROC` は、何を測っていますか。
4. `val_split_mode="from_test"` にした理由は何ですか。
5. `image_AUROC` が 1.0 でも、製品の検査精度として引用してはいけないのはなぜですか。

**English:**

1. Why must abnormal images never appear in `train`?
2. What does PatchCore's "training" actually do? Are weights updated?
3. What do `image_AUROC` and `pixel_AUROC` measure?
4. Why was `val_split_mode="from_test"` used?
5. Why must an `image_AUROC` of 1.0 not be quoted as the inspection accuracy of a product?
#### English: Understanding questions
Model answers: `docs/instructor/day05_instructor_guide.md`.

---

## 困ったときは / Troubleshooting

**日本語：** エラーが出たら、まず「どのセルで」「どの行で」「どんなメッセージが出たか」を 書き留めてください。提出物にも記録します。

**English:** When an error appears, first write down *which cell*, *which line*, and *what the message says*. You will also record it in your submission.

| 症状（日本語） | Symptom (English) | 対処（日本語） | What to do (English) |
| --- | --- | --- | --- |
| `ModuleNotFoundError: No module named 'anomalib'` | `ModuleNotFoundError: No module named 'anomalib'` | カーネルが違います。`Kernel` → `Change Kernel` → `Python 3 (AI Vision Anomalib)`。 | Wrong kernel. Choose `Python 3 (AI Vision Anomalib)`. |
| `AttributeError: 'Folder' object has no attribute 'val_data'` | `AttributeError: 'Folder' object has no attribute 'val_data'` | `val_split_mode="none"` にすると発生します。`from_test` か `from_train` を指定してください。 | Happens with `val_split_mode="none"`. Use `from_test` or `from_train`. |
| `MisMatchError` が出る | `MisMatchError` is raised | 異常画像とマスクの対応がずれています。`ground_truth/<種類>/` のファイル名を確認してください。 | Abnormal images and masks do not pair up. Check the file names under `ground_truth/<type>/`. |
| バックボーンのダウンロードに失敗する | The backbone download fails | `pre_trained=True` は初回に ImageNet の重みを取得します。再実行するか、管理者に相談してください。 | `pre_trained=True` downloads ImageNet weights once. Retry, or ask IT. |
| 学習が異常に遅い | Training is very slow | `wide_resnet50_2` は CPU では非常に遅いです。`resnet18` を使ってください。 | `wide_resnet50_2` is very slow on CPU; use `resnet18`. |
| メモリ不足になる | Out of memory | `train_batch_size` と `eval_batch_size` を 4 に下げてください。 | Lower `train_batch_size` and `eval_batch_size` to 4. |

## 完了チェックリスト / Completion checklist

**日本語：** すべてにチェックが付いたら、このノートブックは完了です。

**English:** This notebook is complete when every box is checked.

- [ ] `aivision-anomalib` カーネルで実行した
      - [ ] Ran with the `aivision-anomalib` kernel
- [ ] `train` が正常のみであることを確認した
      - [ ] Confirmed `train` holds normal images only
- [ ] PatchCore の fit を実行した
      - [ ] Ran the PatchCore fit
- [ ] 4 つの指標を記録した
      - [ ] Recorded the four metrics
- [ ] ヒートマップを確認した
      - [ ] Inspected the heatmaps
- [ ] スコア一覧と予測 JSON を保存した
      - [ ] Saved the score table and prediction JSON
- [ ] 演習 1〜3 を実施した
      - [ ] Completed exercises 1-3
- [ ] 理解度チェックの 5 問に答えた
      - [ ] Answered the 5 questions
- [ ] ノートブックを保存した
      - [ ] Saved the notebook

## さらに学ぶ / Further learning（任意 / optional）

### 参考リンク / Reference links

- [Anomalib ドキュメント](https://anomalib.readthedocs.io/)
- [Anomalib: Folder データモジュール](https://anomalib.readthedocs.io/en/latest/markdown/guides/reference/data/image/folder.html)
- [PatchCore 論文](https://arxiv.org/abs/2106.08265)
- [torchvision のモデル一覧](https://pytorch.org/vision/stable/models.html)

### 追加チャレンジ / Extra challenge（必須ではありません / not mandatory）

**日本語：** `coreset_sampling_ratio` を 0.01 と 0.5 で試し、メモリ使用量と指標の変化を比べてください。

**English:** Try `coreset_sampling_ratio` at 0.01 and 0.5 and compare memory use and metrics.